# SATQUERY AI — SIH26167 Cloud Training & Evaluation

This notebook creates a **real, reproducible BigEarthNet-adapted checkpoint** and held-out evaluation artifact. Use a GPU runtime. It does not fabricate benchmark scores.

In [ ]:
# Colab: Runtime > Change runtime type > GPU. Kaggle: Settings > Accelerator > GPU.
# On Kaggle also enable Internet, or add this repository as a Kaggle Dataset and set REPO_DIR to that input path.
from pathlib import Path
import os, subprocess, sys

IS_KAGGLE = Path('/kaggle').exists()
REPO_DIR = Path('/kaggle/working/osiris' if IS_KAGGLE else '/content/osiris')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/simplifaisoul/osiris.git', str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'ml/remote_sensing_adapter/requirements.txt'], check=True)
subprocess.run(['nvidia-smi'], check=True)
print(f'Working from: {REPO_DIR}')

## Dataset paths

Download BigEarthNet.txt / BigEarthNet only under its licence. Set these paths to your prepared non-overlapping train/test manifests and image root.

In [ ]:
# Kaggle: replace `your-private-bigearthnet` with the exact input-dataset slug shown in the right sidebar.
RAW_DATA_ROOT = Path('/kaggle/input/your-private-bigearthnet') if IS_KAGGLE else Path('/content/drive/MyDrive/bigearthnet')
# Raw BigEarthNet 14K has BigEarthNet-S1, BigEarthNet-S2 and metadata.parquet.
# Convert its Sentinel-2 tiles to RGB PNGs and create separate train/test manifests in a writable directory.
DATA_ROOT = Path('/kaggle/working/bigearthnet_prepared') if IS_KAGGLE else Path('/content/bigearthnet_prepared')
if (RAW_DATA_ROOT / 'metadata.parquet').exists() and not (DATA_ROOT / 'manifests/train.csv').exists():
    subprocess.run(['python', 'ml/remote_sensing_adapter/prepare_bigearthnet14k.py', '--data-root', str(RAW_DATA_ROOT), '--output', str(DATA_ROOT), '--train-limit', '3000', '--test-limit', '500'], check=True)
TRAIN_MANIFEST = DATA_ROOT / 'manifests/train.csv'
TEST_MANIFEST = DATA_ROOT / 'manifests/test.csv'
assert TRAIN_MANIFEST.exists(), f'Missing {TRAIN_MANIFEST}'
assert TEST_MANIFEST.exists(), f'Missing {TEST_MANIFEST}'
print(f'Raw dataset root: {RAW_DATA_ROOT}')
print(f'Prepared training root: {DATA_ROOT}')

In [ ]:
# Validate rows, image files, and train/test leakage before training.
!python ml/remote_sensing_adapter/preflight.py --train "{TRAIN_MANIFEST}" --test "{TEST_MANIFEST}" --image-root "{DATA_ROOT}" --output runs/dataset-preflight.json
!cat runs/dataset-preflight.json

In [ ]:
# Start conservatively; increase batch size only if GPU memory permits.
!python ml/remote_sensing_adapter/train.py --manifest "{TRAIN_MANIFEST}" --image-root "{DATA_ROOT}" --output checkpoints/bigearthnet-clip-adapter.pt --epochs 3 --batch-size 16 --lr 1e-5

In [ ]:
# Held-out adapter evidence; do not call this VRSBench/RSVQA/CDVQA performance.
!python ml/remote_sensing_adapter/evaluate.py --manifest "{TEST_MANIFEST}" --image-root "{DATA_ROOT}" --checkpoint checkpoints/bigearthnet-clip-adapter.pt --output runs/adapter-evaluation.json --limit 500
!cat runs/adapter-evaluation.json
!sha256sum checkpoints/bigearthnet-clip-adapter.pt

## Final SIH evidence

Download the checkpoint and both JSON artifacts. Then run the prescribed VRSBench, RSVQA, and CDVQA evaluators separately. Keep ISRO/SAC data unseen until final evaluation.